# Module of the week: `contextlib`

1. Context managers and `with`
2. Writing a context manager with a class
3. Writing one with `contextlib`
4. stdin and stdout with contextlib
5. A few other things that come with `contextlib`

# What is a context manager?

Just about everyone who uses Python knows that they should use `with` when opening files. That guarantees that by the time you finish with the `with` block, the file is flushed and closed.

In [1]:
for one_line in open('/etc/passwd'):  # is this file ever closed?
    print(len(one_line), end=' ')

3 16 3 76 71 18 2 70 18 3 59 50 54 72 62 64 70 61 71 70 70 72 56 66 62 67 52 63 60 69 58 50 50 54 66 67 59 63 64 61 62 61 62 61 55 55 74 53 65 65 55 56 50 56 88 66 61 70 81 65 62 56 75 65 54 64 75 72 85 72 67 53 55 69 77 74 94 85 97 73 84 68 71 70 63 55 82 74 64 66 76 55 78 80 56 63 82 76 63 57 69 61 99 73 55 63 79 100 57 83 62 77 104 55 67 92 89 64 62 51 76 74 75 84 52 80 77 85 62 78 82 73 77 76 75 97 104 61 69 77 85 53 

In [4]:
# what does it look like to use with?

with open('/etc/passwd') as f:    # this is like assigning f, the variable, a value -- the file object we get from open
    # with automatically runs f.__enter__() just before starting the block

    for one_line in f:
        print(len(one_line), end=' ')

    # with automatically runs f.__exit__() just after ending the block


# these methods can do anything, but they're usually doing some setup/teardown
# in the case of a file, __exit__ flushes and closes the file

3 16 3 76 71 18 2 70 18 3 59 50 54 72 62 64 70 61 71 70 70 72 56 66 62 67 52 63 60 69 58 50 50 54 66 67 59 63 64 61 62 61 62 61 55 55 74 53 65 65 55 56 50 56 88 66 61 70 81 65 62 56 75 65 54 64 75 72 85 72 67 53 55 69 77 74 94 85 97 73 84 68 71 70 63 55 82 74 64 66 76 55 78 80 56 63 82 76 63 57 69 61 99 73 55 63 79 100 57 83 62 77 104 55 67 92 89 64 62 51 76 74 75 84 52 80 77 85 62 78 82 73 77 76 75 97 104 61 69 77 85 53 

In [5]:
with open('myfile.txt', 'w') as f:
    f.write('abcd\n')
    f.write('efgh\n')

In [6]:
# I know that those lines have been written to the file, because the "with" block guarantees flushing + closing



In [7]:
!cat myfile.txt

abcd
efgh


# Context manager protocol

A context manager is an object that knows how to behave inside of a `with` block:

- It defines `__enter__` -- this method returns the new object
- It defines `__exit__` -- this method gets arguments passed if there is an exception

# What else uses this protocol?

- `threading.Lock` -- you can use it with `with`, and then lock/release a critical region


In [8]:
# how can I add this to a class I'm writing?

class MyCM:
    def __enter__(self):
        print('\tNow in __enter__!')

    def __exit__(self, type, value, tb):
        print('\tNow in __exit__!')

with MyCM() as m:
    print('hello')

	Now in __enter__!
hello
	Now in __exit__!


In [12]:
# One common case: I want to temporarily reassign sys.stdout to another file

import sys

class RedirectStdout:
    def __init__(self, stdout_filename):
        self.real_stdout = sys.stdout
        self.stdout_filename = stdout_filename

    def __enter__(self):
        self.stdout_file = open(self.stdout_filename, 'w')
        sys.stdout = self.stdout_file

    def __exit__(self, type, value, tb):
        sys.stdout = self.real_stdout    # get the OG back
        self.stdout_file.close()          # flush + close what we were using

In [13]:
print('Before')
with RedirectStdout('mystdout.txt') as rs:
    print('hello')
print('After')    

Before
After


In [14]:
!cat mystdout.txt

hello


# Contextlib provides tools for easier context managers

If we use `contextlib.contextmanager` as a decorator on a generator function:

- The generator function works as we would expect
- Everything before the `yield` is the `__enter__`
- Everything after the `yield` is the `__exit__`

In [15]:
import contextlib

@contextlib.contextmanager
def redirect_stdout(stdout_filename):
    real_stdout = sys.stdout

    stdout_file = open(stdout_filename, 'w')
    sys.stdout = stdout_file

    yield

    sys.stdout = real_stdout
    stdout_file.close()



In [16]:
print('Before')
with redirect_stdout('function_mystdout.txt') as rs:
    print('hello from the function')
print('After')    

Before
After


In [17]:
!cat function_mystdout.txt

hello from the function


In [18]:
print('Before')
with redirect_stdout('function_mystdout.txt') as rs:
    raise ValueError('bad news!')
print('After')    

ValueError: bad news!

Before


In [19]:
import contextlib

@contextlib.contextmanager
def redirect_stdout(stdout_filename):
    real_stdout = sys.stdout

    stdout_file = open(stdout_filename, 'w')
    sys.stdout = stdout_file

    try:
        yield
    
        sys.stdout = real_stdout
        stdout_file.close()
    except Exception as e:
        print(f'We got an exception: {e}')


In [20]:
print('Before')
with redirect_stdout('function_mystdout.txt') as rs:
    raise ValueError('bad news!')
print('After')    

In [21]:
print('a')

In [3]:
import sys
import contextlib

@contextlib.contextmanager
def redirect_stdout(stdout_filename):
    real_stdout = sys.stdout

    stdout_file = open(stdout_filename, 'w')
    sys.stdout = stdout_file

    try:
        yield
    
    except Exception as e:
        print(f'We got an exception: {e}')   # this goes into our temp stdout, because it's still sys.stdout!
    finally:
        sys.stdout = real_stdout
        stdout_file.close()

In [4]:
print('Before')
with redirect_stdout('function_mystdout.txt') as rs:
    raise ValueError('bad news!')
print('After')    

Before
After


In [5]:
print('a')

a


In [8]:
with open('my_new_stdout.txt', 'w') as f:
    rs = contextlib.redirect_stdout(f)

    with rs:
        print('Hello')    

In [9]:
!cat my_new_stdout.txt

Hello


In [10]:
with open('my_new_stdout.txt', 'w') as f:
    rs = contextlib.redirect_stdout(f)

    with rs:
        print('a')
        raise ValueError('bad news!')
        print('b')

ValueError: bad news!

In [15]:
# what if I want to open a file, and I know that it might fail, and if it fails, I'm OK with doing nothing.

filename = input('Enter a filename: ')

with contextlib.suppress(FileNotFoundError):
    with open(filename) as f:
        size = len(f.read())
        print(f'{f.name} has {size} characters.')

Enter a filename:  asdfasddfafafa


In [17]:
# we can combine the withs

filename = input('Enter a filename: ')

with (contextlib.suppress(FileNotFoundError), 
      open(filename) as f):
    size = len(f.read())
    print(f'{f.name} has {size} characters.')

Enter a filename:  /etc/passwd


/etc/passwd has 9344 characters.
